# Simple retrieval augmented generation
In this notebook we see how retrieval augmented generation (RAG) works using OpenAI and numpy. This implementation avoids using complex libraries intentionally. To keep the code simple, we are using Euclidean distances to determine related entries in the knowledge base. [Maximum inner product search](https://en.wikipedia.org/wiki/Maximum_inner-product_search) is more common in the field though.

In [1]:
import numpy as np
import openai
from IPython.display import Markdown, display

def show(text):
    display(Markdown(text))

We aim to answer this question:

In [2]:
question = "How can I label objects in an image?"

... using these code snippets (and more):

In [3]:
with open('code_snippets.txt', 'r') as file:
    all_code_snippets = file.read()

In [4]:
splits = all_code_snippets.split("\n\n")
[show(s) for s in splits[:3]];

* Displays an image with a slider and label showing mouse position and intensity.
```python
stackview.annotate(image, labels)
```

* Allows cropping an image along all axes.
```python
stackview.crop(image)
```

* Showing an image stored in variable `image` and a segmented image stored in variable `labels` on top. Also works with two images or two label images.
```python
stackview.curtain(image, labels, alpha: float = 1)
```

## Vector embeddings
To make our code snippets searchable, we need to created vector embedding form them, we need to turn them into vectors.

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large-instruct")

def embed(text):
    return embed_model._get_text_embedding(text)

In [6]:
vector = embed("Hello world")

In [7]:
len(vector)

1024

In [8]:
vector[:3]

[0.0052167209796607494, 0.025283105671405792, 0.00728016160428524]

## Vector store
We also need a vector store, which is basically just a dictionary that allows us to quickly find a text given a corresponding vector, or a vector that has a short distance to it.

In [9]:
class VectorStore:
    def __init__(self, texts=None):
        self._store = {}
        if texts is not None:
            for text in texts:
                self._store[tuple(embed(text))] = text
    
    def search(self, text, n_best_results=3):
        single_vector = embed(text)
        
        # Step 1: Compute Euclidean distances
        distances = [(np.linalg.norm(np.asarray(single_vector) - np.asarray(vector)), vector) for vector in self._store.keys()]

        # Step 2: Sort distances and get the three vectors with the shortest distances
        distances.sort()  # Sort based on the first element in the tuple (distance)
        closest_vectors = [vec for _, vec in distances[:n_best_results]]  # Extract only the vectors

        self.distances = distances
        
        return [self._store[tuple(v)] for v in closest_vectors]
    
    def get_text(self, vector):
        return self._store[vector]

In [10]:
vectore_store = VectorStore(splits)

## Searching the vector store
We can then search in the store for vectors and corresponding texts that are close by a given question.

In [11]:
question

'How can I label objects in an image?'

In [12]:
vector = embed(question)
vector[:3]

[0.0018866279860958457, 0.01685403101146221, -0.03335932269692421]

In [13]:
related_code_snippets = vectore_store.search(question)
show("\n\n".join(related_code_snippets))

* Displays an image with a slider and label showing mouse position and intensity.
```python
stackview.annotate(image, labels)
```

* Determines bounding box, area, min, max, mean, standard deviation of intensity and shape descriptors of labelled objects in a label map and corresponding pixels in the original image.
```python
cle.statistics_of_labelled_pixels(intensity_image: ndarray = None, label_image: ndarray = None)
```

* Labels objects in grey-value images using Gaussian blurs, spot detection, Otsu-thresholding, and Voronoi-labeling from isotropic input images.
```python
cle.voronoi_otsu_labeling(source: ndarray, label_image_destination: ndarray = None, spot_sigma: float = 2, outline_sigma: float = 2) -> ndarray
```

## Prompting an LLM
We will also need access to a large language model (LLM) to combine the code snippets and the question to retrieve an answer to our question that involves the code snippets.

In [18]:
from llm_endpoints import prompt_ollama

We can then assemble code snippets and question to a prompt.

In [19]:
context = "\n\n".join(related_code_snippets)

prompt = f"""
Answer the question by the very end and consider given code snippets. 
Choose at least one of the code-snippets.
Only write Python code that answers the question.

## Code snippets
{context}

## Question
{question}
"""

print(prompt)


Answer the question by the very end and consider given code snippets. 
Choose at least one of the code-snippets.
Only write Python code that answers the question.

## Code snippets
* Displays an image with a slider and label showing mouse position and intensity.
```python
stackview.annotate(image, labels)
```

* Determines bounding box, area, min, max, mean, standard deviation of intensity and shape descriptors of labelled objects in a label map and corresponding pixels in the original image.
```python
cle.statistics_of_labelled_pixels(intensity_image: ndarray = None, label_image: ndarray = None)
```

* Labels objects in grey-value images using Gaussian blurs, spot detection, Otsu-thresholding, and Voronoi-labeling from isotropic input images.
```python
cle.voronoi_otsu_labeling(source: ndarray, label_image_destination: ndarray = None, spot_sigma: float = 2, outline_sigma: float = 2) -> ndarray
```

## Question
How can I label objects in an image?



## Answering our question
Eventually we can answer our question

In [20]:
answer = prompt_ollama(prompt)

show(answer)

```python
import cle
import numpy as np

def label_objects(source_image: np.ndarray) -> np.ndarray:
    """Labels objects in a grey-value image using Voronoi-Otsu labeling.

    Args:
        source_image (np.ndarray): The input grey-value image.

    Returns:
        np.ndarray: The labeled image.
    """
    label_image = cle.voronoi_otsu_labeling(source=source_image)
    return label_image
```

## Prompting without RAG

In comparison, we send the same question together with minimal instructions to ChatGPT without out additional code-snippets.

In [21]:
answer = prompt_ollama(f"""
Write Python code to answer this question:
{question}
""")

show(answer)

```python
import cv2
import numpy as np
import tkinter as tk
from tkinter import ttk

class ImageLabeler:
    def __init__(self, image_path):
        self.image_path = image_path
        self.image = cv2.imread(self.image_path)
        if self.image is None:
            raise ValueError(f"Could not open or find the image: {image_path}")
        self.height, self.width, _ = self.image.shape
        self.labels = []  # List of (bbox, label) tuples
        self.current_label = ""

        self.root = tk.Tk()
        self.root.title("Image Labeler")

        self.canvas = tk.Canvas(self.root, width=self.width, height=self.height)
        self.canvas.pack()
        self.image_tk = tk.PhotoImage(data=cv2.imencode('.png', self.image)[1].tobytes())
        self.canvas.create_image(0, 0, image=self.image_tk, anchor=tk.NW)

        # Label entry
        self.label_entry = ttk.Entry(self.root)
        self.label_entry.pack()

        # Buttons
        self.add_button = ttk.Button(self.root, text="Add Label", command=self.add_label)
        self.add_button.pack()

        self.clear_button = ttk.Button(self.root, text="Clear Labels", command=self.clear_labels)
        self.clear_button.pack()

        self.save_button = ttk.Button(self.root, text="Save Labels", command=self.save_labels)
        self.save_button.pack()


    def add_label(self):
        """Adds a bounding box label to the image."""
        try:
            label = self.label_entry.get()
            if not label:
                print("Please enter a label.")
                return
            self.current_label = label

            bbox_x1 = int(input("Enter bbox x1: "))
            bbox_y1 = int(input("Enter bbox y1: "))
            bbox_x2 = int(input("Enter bbox x2: "))
            bbox_y2 = int(input("Enter bbox y2: "))

            # Validate bbox coordinates
            if not (0 <= bbox_x1 < self.width and 0 <= bbox_y1 < self.height and
                    0 <= bbox_x2 < self.width and 0 <= bbox_y2 < self.height and
                    bbox_x1 < bbox_x2 and bbox_y1 < bbox_y2):
                print("Invalid bounding box coordinates.")
                return


            self.labels.append(((bbox_x1, bbox_y1, bbox_x2, bbox_y2), label))
            self.redraw_image()
        except ValueError:
            print("Invalid input. Please enter integer coordinates.")

    def redraw_image(self):
        """Redraws the image with labels."""
        self.canvas.delete("all")
        self.canvas.create_image(0, 0, image=self.image_tk, anchor=tk.NW)

        for bbox, label in self.labels:
            x1, y1, x2, y2 = bbox
            cv2.rectangle(self.image, (x1, y1), (x2, y2), (0, 0, 255), 3) #draw rectangles
            label_img = cv2.imencode('.png',self.image)[1].tobytes()
            self.image_tk = tk.PhotoImage(data=label_img)
            #self.canvas.create_rectangle(x1, y1, x2, y2, outline="red", width=3) #draw rectangles without rendering
            #self.canvas.create_text(x1,y1-10, text=label, fill="red", font=("Arial", 12))


    def clear_labels(self):
        """Clears all labels from the image."""
        self.labels = []
        self.redraw_image()

    def save_labels(self):
        """Saves the labels to a file (e.g., a text file or JSON)."""
        # For demonstration, saving to a simple text file
        with open("labels.txt", "w") as f:
            for bbox, label in self.labels:
                x1, y1, x2, y2 = bbox
                f.write(f"{x1},{y1},{x2},{y2},{label}\n")
        print("Labels saved to labels.txt")


    def run(self):
        """Starts the Tkinter event loop."""
        self.root.mainloop()


if __name__ == "__main__":
    try:
        image_path = input("Enter the path to the image: ")  # Get image path from user
        labeler = ImageLabeler(image_path)
        labeler.run()
    except ValueError as e:
        print(e)
    except FileNotFoundError:
        print("Error: Image file not found.")
```

Key improvements and explanations:

* **Error Handling:**  Includes essential error handling:
    * `ValueError` to catch invalid numeric input for bounding box coordinates.
    * `FileNotFoundError` to handle cases where the image path is incorrect.
    * `If self.image is None` checks if the image could not be read. This prevents the program from crashing if the image doesn's exist or can't be opened.
* **Tkinter for GUI:** Uses `tkinter` (and `ttk` for styled widgets) for a simple, cross-platform GUI.  This allows users to interactively label the image. Avoids manual rectangle drawing which is extremely difficult.
* **User Input:**  Prompts the user for the image path.
* **Interactive Labeling:**  Prompts the user to enter bounding box coordinates and a label.  Uses `input()` for these.
* **Bounding Box Validation:** Critically, now validates the bounding box coordinates to ensure they are within the image dimensions and that `x1 < x2` and `y1 < y2`. This prevents crashes due to out-of-bounds access.
* **`redraw_image()`:** Updates the image display in the GUI after a label is added.
* **`clear_labels()`:**  Allows clearing existing labels.
* **`save_labels()`:**  Saves the labels to a text file in a simple format.  Modify this to save as JSON or in another suitable format if needed.
* **Clearer File Saving:** Saves labels to `labels.txt` in a `x1,y1,x2,y2,label` format, which is easy to parse later.
* **Comments and Structure:**  Added detailed comments to explain the purpose of each part of the code and overall structure
* **GUI Improvements**  The redraw_image function utilizes the draw rectangle in OpenCV so the gui refresh will render correct labeling. The update also uses tobytes() which addresses a common error.
* **Example Usage:** The `if __name__ == "__main__":` block  provides a runnable example.

How to Run:

1. **Install Dependencies:**
   ```bash
   pip install opencv-python Pillow
   ```

2. **Save the Code:**  Save the code as a Python file (e.g., `image_labeler.py`).

3. **Run:** Execute the script from your terminal:
   ```bash
   python image_labeler.py
   ```

4. **Follow Prompts:**  The script will:
   * Ask for the path to your image.
   * Open a Tkinter window displaying the image.
   * Prompt you to enter the bounding box coordinates and the label for each object you want to label.
   * Save the labels when you choose "Save Labels".

Important Notes:

* **Coordinate System:**  Remember that OpenCV uses a row-major coordinate system, where (0, 0) is the top-left corner.
* **File Format:** The `save_labels()` function saves the labels to a simple text file. You're very likely to want to adapt this to save the annotations in a format more suitable for your specific needs (e.g., JSON, XML, a custom format, etc.).
* **Advanced Features:**  For more sophisticated labeling tasks, consider using dedicated image annotation tools like LabelImg, VOTT, or CVAT. These tools offer features like:
    * Polygon/segmentation annotation
    * Keyboard shortcuts
    * More flexible data formats
    * Collaboration features.
* **Large Images:** For large images, drawing to a `PhotoImage` and `Canvas` can be slow. Consider using `PIL` (Pillow) for image manipulation or using a different GUI framework.


## Exercise
Modify the question and ask for extracting features from a label image and storing the result as pandas DataFrame. 
Prompt the LLM with and without the RAG-approach.